# Fine-tuning NLLB-200 sur le bamoun (bamoun ⇄ français)

**Avant de commencer**
1. Menu *Exécution → Modifier le type d'exécution* : choisis **GPU (T4)**.
2. Lance les cellules **dans l'ordre**, une par une.
3. Si une cellule échoue, copie le message d'erreur en entier.

In [ ]:
!pip install -q transformers sentencepiece sacrebleu accelerate
import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "AUCUN : active le GPU !")

GPU : Tesla T4


## 1. Connecter Google Drive et charger les fichiers
Tout ce qui compte (données, tokenizer, points de sauvegarde, résultats) est enregistré dans le dossier `nllb_bamoun` de ton Drive. Si Colab s'arrête, **relance simplement les cellules dans l'ordre** : les données ne sont pas redemandées et l'entraînement reprend là où il s'était arrêté.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
BASE = "/content/drive/MyDrive/nllb_bamoun"
DATA = f"{BASE}/data"
os.makedirs(DATA, exist_ok=True)

noms = ["bamoun_train.tsv", "bamoun_dev.tsv", "bamoun_test.tsv"]
if not all(os.path.exists(f"{DATA}/{n}") for n in noms):
    from google.colab import files
    print("Uploade bamoun_train.tsv (ou ton fichier CSV de corpus), bamoun_dev.tsv et bamoun_test.tsv")
    uploaded = files.upload()

    # Si le fichier train a été uploadé sous un autre nom (ex: bamoun_corpus_propre (1).csv)
    for nom_fich in uploaded.keys():
        if "corpus" in nom_fich.lower() or "propre" in nom_fich.lower() or nom_fich.endswith(".csv"):
            print(f"Renommage de {nom_fich} en bamoun_train.tsv...")
            shutil.move(nom_fich, "bamoun_train.tsv")

    for n in noms:
        if not os.path.exists(n):
            raise FileNotFoundError(f"Le fichier requis '{n}' n'a pas été trouvé. Assure-toi de l'avoir bien téléversé.")
        shutil.copy(n, f"{DATA}/{n}")

for n in noms:
    shutil.copy(f"{DATA}/{n}", n)
print("Données prêtes (copie conservée dans Drive).")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Uploade bamoun_train.tsv (ou ton fichier CSV de corpus), bamoun_dev.tsv et bamoun_test.tsv


Saving bamoun_test.tsv to bamoun_test (1).tsv
Saving bamoun_dev.tsv to bamoun_dev (1).tsv
Saving bamoun_train.tsv to bamoun_train.tsv
Données prêtes (copie conservée dans Drive).


In [ ]:
import csv, pandas as pd

def lire(path):
    return pd.read_csv(path, sep="\t", header=None, names=["bax", "fra"],
                       quoting=csv.QUOTE_NONE, escapechar="\\", dtype=str, keep_default_na=False)

train, dev, test = lire("bamoun_train.tsv"), lire("bamoun_dev.tsv"), lire("bamoun_test.tsv")
print(len(train), "entraînement |", len(dev), "validation |", len(test), "test")
print(train.head(3).to_string())

1569 entraînement | 182 validation | 200 test
                                                            bax                                                     fra
0    Euh mí u tóóshə́ ŋwət ru, nzíé yúá yʉ́ə́ u púá' yírə́ nə́.               Que tu te présentes, tu dis ce que tu es.
1                                                Í nzie Li shá?                             Veut-il parler de mon nom ?
2  Li shú, nə nguu yúá u púá' yírə́ nə́, nə́ nguu yúá u yí nə́.  Ton nom, et tout ce que tu es, et tout ce que tu sais.


## 2. Modèle et nouveau code de langue
Le bamoun n'a pas de code dans NLLB. On ajoute `bax_Latn` et on copie le plongement d'une langue bantoue proche déjà connue (`lin_Latn`, lingala) comme point de départ.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL    = "facebook/nllb-200-distilled-600M"
NEW_CODE = "bax_Latn"     # bamoun (ISO 639-3 : bax), écriture latine
REF_CODE = "lin_Latn"     # langue proche pour initialiser le nouveau code
FR_CODE  = "fra_Latn"

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
assert tok.convert_tokens_to_ids(REF_CODE) != tok.unk_token_id, "REF_CODE inconnu : essaie kin_Latn"

tok.add_tokens([NEW_CODE], special_tokens=True)
new_id = tok.convert_tokens_to_ids(NEW_CODE)
if len(tok) > model.get_input_embeddings().num_embeddings:
    model.resize_token_embeddings(len(tok))
with torch.no_grad():
    E = model.get_input_embeddings().weight
    E[new_id] = E[tok.convert_tokens_to_ids(REF_CODE)].clone()
print("id du code bamoun :", new_id)

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

id du code bamoun : 256204


### Vérification importante
La première ligne doit afficher le code de langue **en premier** dans les deux listes. Si l'`assert` échoue, envoie-moi l'affichage.

In [ ]:
tok.src_lang, tok.tgt_lang = NEW_CODE, FR_CODE
ex = tok("Mí u tóóshə́ ŋwət ru.", text_target="Que tu te présentes.")
print(tok.convert_ids_to_tokens(ex["input_ids"]))
print(tok.convert_ids_to_tokens(ex["labels"]))
assert ex["input_ids"][0] == new_id and ex["labels"][0] == tok.convert_tokens_to_ids(FR_CODE)

ids = tok(train.bax.tolist(), add_special_tokens=False)["input_ids"]
n = sum(len(x) for x in ids); unk = sum(i == tok.unk_token_id for x in ids for i in x)
print(f"{100*unk/n:.2f}% de tokens inconnus (côté bamoun)")

tok.save_pretrained(f"{BASE}/tokenizer")   # conserve le nouveau code de langue dans Drive

['bax_Latn', '▁Mí', '▁u', '▁tó', 'ó', 'sh', 'ə', '́', '▁ŋ', 'w', 'ət', '▁ru', '.', '</s>']
['fra_Latn', '▁Que', '▁tu', '▁te', '▁prés', 'entes', '.', '</s>']
0.00% de tokens inconnus (côté bamoun)


('/content/drive/MyDrive/nllb_bamoun/tokenizer/tokenizer_config.json',
 '/content/drive/MyDrive/nllb_bamoun/tokenizer/tokenizer.json')

## 3. Préparer les données (dans les deux sens)

In [ ]:
MAXLEN = 160

class Paires(torch.utils.data.Dataset):
    def __init__(self, df):
        self.items = []
        for src, tgt, sl, tl in [("bax", "fra", NEW_CODE, FR_CODE), ("fra", "bax", FR_CODE, NEW_CODE)]:
            tok.src_lang, tok.tgt_lang = sl, tl
            enc = tok(df[src].tolist(), text_target=df[tgt].tolist(), max_length=MAXLEN, truncation=True)
            for i in range(len(df)):
                self.items.append({"input_ids": enc["input_ids"][i],
                                   "attention_mask": enc["attention_mask"][i],
                                   "labels": enc["labels"][i]})
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

train_ds, dev_ds = Paires(train), Paires(dev)
print(len(train_ds), "exemples d'entraînement,", len(dev_ds), "de validation")

3138 exemples d'entraînement, 364 de validation


## 4. Entraînement (avec sauvegardes sur Drive et reprise automatique)
Compte de 30 à 90 minutes sur un T4. Un point de sauvegarde est écrit sur ton Drive **toutes les 200 étapes** (environ une fois par époque). Si Colab s'arrête, relance les cellules : l'entraînement repart du dernier point de sauvegarde.
En cas d'erreur de mémoire (*CUDA out of memory*) : mets `per_device_train_batch_size=4` et `gradient_accumulation_steps=4`.

In [ ]:
import glob
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, EarlyStoppingCallback)

CKPT_DIR = f"{BASE}/checkpoints"
collator = DataCollatorForSeq2Seq(tok, model=model, label_pad_token_id=-100)
cfg = dict(output_dir=CKPT_DIR, per_device_train_batch_size=8, per_device_eval_batch_size=16,
           gradient_accumulation_steps=2, learning_rate=1e-4, optim="adafactor",
           num_train_epochs=10, warmup_steps=100, fp16=torch.cuda.is_available(),
           save_strategy="steps", save_steps=200, eval_steps=200, save_total_limit=2,
           load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
           logging_steps=50, report_to="none")
try:
    args = Seq2SeqTrainingArguments(eval_strategy="steps", **cfg)
except TypeError:
    args = Seq2SeqTrainingArguments(evaluation_strategy="steps", **cfg)

trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_ds, eval_dataset=dev_ds,
                         data_collator=collator,
                         callbacks=[EarlyStoppingCallback(early_stopping_patience=3)])

ckpts = sorted(glob.glob(f"{CKPT_DIR}/checkpoint-*"), key=lambda p: int(p.split("-")[-1]))
if ckpts:
    print("Reprise depuis :", ckpts[-1])
    trainer.train(resume_from_checkpoint=ckpts[-1])
else:
    trainer.train()

Step,Training Loss,Validation Loss
200,5.008414,2.337531
400,3.978416,2.004173
600,3.269031,1.887574
800,2.593464,1.824224
1000,2.152108,1.799807
1200,1.853462,1.816725
1400,1.652513,1.836987
1600,1.395387,1.838434


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


**Si la reprise échoue** (par exemple parce que Colab s'est arrêté pendant l'écriture d'un point de sauvegarde), supprime le dossier `checkpoint-…` au plus grand numéro dans `nllb_bamoun/checkpoints` sur ton Drive, puis relance la cellule : elle repartira du point précédent.

## 5. Évaluation (chrF++) sur le jeu de test
Le jeu de test n'a jamais été vu pendant l'entraînement. Le « point de départ » recopie simplement la phrase d'origine : c'est le score à battre.

In [ ]:
try:
    import sacrebleu
except ModuleNotFoundError:
    print("Installation de sacrebleu manquante...")
    !pip install -q sacrebleu
    import sacrebleu

def traduire_lot(textes, src_code, tgt_code, beams=4, bs=16):
    tok.src_lang = src_code
    model.eval(); sorties = []
    for i in range(0, len(textes), bs):
        batch = tok(textes[i:i+bs], return_tensors="pt", padding=True,
                    truncation=True, max_length=MAXLEN).to(model.device)
        long_max = int(batch["input_ids"].shape[1] * 1.5) + 10      # une traduction ne doit pas être bien plus longue que la source
        with torch.no_grad():
            out = model.generate(**batch,
                                 forced_bos_token_id=tok.convert_tokens_to_ids(tgt_code),
                                 num_beams=beams, max_new_tokens=min(long_max, MAXLEN),
                                 no_repeat_ngram_size=4, repetition_penalty=1.1)
        sorties += tok.batch_decode(out, skip_special_tokens=True)
    return sorties

def chrf(hyps, refs):
    return sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score

for nom, src, tgt, sc, tc in [("bamoun → français", "bax", "fra", NEW_CODE, FR_CODE),
                              ("français → bamoun", "fra", "bax", FR_CODE, NEW_CODE)]:
    hyps = traduire_lot(test[src].tolist(), sc, tc)
    pd.DataFrame({"source": test[src], "modele": hyps, "attendu": test[tgt]}).to_csv(
        f"{BASE}/resultats_test_{src}_vers_{tgt}.csv", index=False, encoding="utf-8-sig")
    print(f"{nom} : chrF++ modèle = {chrf(hyps, test[tgt].tolist()):.1f} | point de départ (copie) = {chrf(test[src].tolist(), test[tgt].tolist()):.1f}")
    for s, h, r in list(zip(test[src], hyps, test[tgt]))[:6]:
        print("  SOURCE  :", s); print("  MODÈLE  :", h); print("  ATTENDU :", r); print()

Installation de sacrebleu manquante...


NameError: name 'NEW_CODE' is not defined

## 6. Sauvegarder le modèle final (dans ton Drive)

In [ ]:
DEST = f"{BASE}/final"
model.save_pretrained(DEST); tok.save_pretrained(DEST)
print("Sauvegardé dans", DEST)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Sauvegardé dans /content/drive/MyDrive/nllb_bamoun/final


## 7. Utilisation
Pour brancher ce modèle dans `traduction_securisee.py`, passe-lui une fonction de ce type :

In [ ]:
def fr_vers_bamoun(texte):
    return traduire_lot([texte], FR_CODE, NEW_CODE)[0]

def bamoun_vers_fr(texte):
    return traduire_lot([texte], NEW_CODE, FR_CODE)[0]

print(fr_vers_bamoun("comment vas-tu ?"))
print(bamoun_vers_fr("Mí u tóóshə́ ŋwət ru"))
print(fr_vers_bamoun("votre solde est de"))
print(fr_vers_bamoun("je veux un café"))


[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


U púá' yie kʉə ?


[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Que tu t'enfuyais


[transformers] Both `max_new_tokens` (=160) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Fʉ́'shə́ shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́kɛ́t shʉ́t shʉ́kɛ́t shʉ́t
Mə́ fá' káfe.


In [ ]:
!pip install -q gradio
import gradio as gr

def interface(texte, direction):
    if direction == "Bamoun → Français":
        return traduire_lot([texte], NEW_CODE, FR_CODE)[0]
    return traduire_lot([texte], FR_CODE, NEW_CODE)[0]

demo = gr.Interface(
    interface,
    [gr.Textbox(label="Texte", lines=3),
     gr.Radio(["Bamoun → Français", "Français → Bamoun"], value="Bamoun → Français", label="Direction")],
    gr.Textbox(label="Traduction"),
    title="Traducteur bamoun (prototype)")
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://73bf4d628af5034199.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
DEST = "/content/drive/MyDrive/nllb_bamoun/final"
tok = AutoTokenizer.from_pretrained(DEST)
model = AutoModelForSeq2SeqLM.from_pretrained(DEST).to("cuda" if torch.cuda.is_available() else "cpu")
NEW_CODE, FR_CODE, MAXLEN = "bax_Latn", "fra_Latn", 160

OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/content/drive/MyDrive/nllb_bamoun/final'. Use `repo_type` argument if needed.